# Baseline BiLSTM (siamese, embedding word-piece MiniLM đóng băng) – GPU

**Settings → Accelerator: GPU T4 x2 hoặc P100**, **Internet: On**. Không cần khóa API.

Bấm **Save Version → Save & Run All**. Chạy lần lượt 3 thiết lập: bộ đề tài chia theo JD, loại trừ hai chiều, resume-job-fit (5 seed mỗi thiết lập).

Xong thì tải mọi file `bilstm_*.csv` ở tab Output (6 file), đặt vào `report/results/` trên máy. File được ghi sau mỗi fold, nên nếu phiên dừng giữa chừng vẫn tải được phần đã chạy.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b bang-branch https://github.com/hoangphungvan/doantotnghiep.git
%cd doantotnghiep
!git log --oneline -1

In [ ]:
# Chỉ cài phần còn thiếu, KHÔNG cài lại torch (giữ bản CUDA của Kaggle)
!pip install -q torch_geometric python-dotenv openai pymupdf
import torch, sentence_transformers; print(torch.__version__, torch.cuda.is_available(), sentence_transformers.__version__)

In [ ]:
# Bộ dữ liệu không nằm trong repo: tải train.csv/test.csv gốc từ HuggingFace
from huggingface_hub import snapshot_download
snapshot_download('cnamuangtoun/resume-job-description-fit', repo_type='dataset', local_dir='data/raw/cnamuangtoun')
!ls -la data/raw/cnamuangtoun

In [ ]:
# 5 seed x 5 fold (double: 10 ô), cùng fold/validation/seed với mô hình chính. Dòng đầu mỗi lượt in số cặp (1664/1664/7993)
!mkdir -p /kaggle/working
for p in ['jd', 'double', 'ext']:
    !python -X utf8 src/modeling/train_bilstm.py {p} --seeds 5 2>&1 | grep -v Warning
    !cp report/results/bilstm_*.csv /kaggle/working/

In [ ]:
!cp report/results/bilstm_*.csv /kaggle/working/ 2>/dev/null; ls -la /kaggle/working/*.csv